In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.gridspec as gridspec
import openpyxl
import seaborn as sns
import scipy.stats as stats
import math
import numpy_financial as npf
import pyxirr 

from collections import defaultdict       # library to create dictionarie variables

from IPython.display import display, HTML, Markdown

#### Function to import range

In [1]:
def Import_Range_from_Excel(sheet,range_str,row_i):
    cell_range = sheet[range_str] # Define cell range (e.g., B2:E10)
    
    # Convert range to a pandas DataFrame
    data = []
    for row in cell_range: data.append([cell.value for cell in row])
    
    df = pd.DataFrame(data)
    
    # Optional: Set the first row as the header
    #df.columns = df.iloc[row_i]
    #df = df[1:].reset_index(drop=True)
    
    df.head()
    return df  # Optional

In [3]:
def Print_3_txt(x,y,z,n_spaces,var_type):
    from IPython.display import display, HTML, Markdown, display_html

    length = len(x)
    if length > n_spaces: length = n_spaces
    display(HTML(f"<b> {x + ("&nbsp;" * (n_spaces-length))} :</b> <font size='3'>{y} </font> <font size='1'>{z}</font>"))
    
    if var_type == True:
        print('Tipo de variable:', type(y))
    if len(str(y))>0: print('')

### Generate random values based on input

In [10]:
def Generate_rand_val(var_name, Input_var_dict,my_dict_input_vals, test_val=0):
    import numpy as np
    import scipy.stats as stats
    
    i_mean, i_SD = Input_var_dict[var_name][2], Input_var_dict[var_name][3]
    i_Low_limit, i_High_limit = Input_var_dict[var_name][0], Input_var_dict[var_name][1]
    i_constant = Input_var_dict[var_name][4]
    i_x_Label, i_Title = Input_var_dict[var_name][6], Input_var_dict[var_name][6]
    i_Dist_Type = Input_var_dict[var_name][5]
        
    ## generate data for truncated normal distribution
    if i_Dist_Type == 'Constant':
        NT_Samples = i_constant
        
    if i_Dist_Type == 'Uniform':
        NT_Samples = np.random.uniform(low = i_Low_limit, high = i_High_limit, size = 1)
        
    if i_Dist_Type == 'Normal':
        NT_Samples = stats.truncnorm.rvs((i_Low_limit - i_mean)/i_SD,(i_High_limit-i_mean)/i_SD, loc=i_mean, scale=i_SD, size=1)
    
    if i_Dist_Type == 'Triangular':
        NT_Samples = np.random.triangular(left=i_Low_limit, mode=i_mean, right= i_High_limit, size=1)
    
    #print(NT_Samples)
    if NT_Samples < 1:
        NT_Samples = np.around(NT_Samples,4)
    else:
        NT_Samples = np.around(NT_Samples,1)
    
    ## Save input values in dictionary
    if i_Dist_Type == 'Constant':    
        my_dict_input_vals[var_name].extend([NT_Samples])
    else:
        my_dict_input_vals[var_name].extend(NT_Samples)   #Save input values in dictionary
    
    return NT_Samples, my_dict_input_vals

### Calculate NPV + IRR

In [7]:
def Calc_Eco_Evalv2(Prod_df, GasPrice, OilPrice,Gob_Tax):

    
    
    BTU_scf = 1050
    USD_MMBTU = GasPrice
    Oil_Price = OilPrice
    Gov_tax = Gob_Tax
    Company_Share = 0.5
    Headco_cost = 1000000
    OPEX = -5
    decommissioning = -4000000

    Cash_Flow_df = pd.DataFrame()
    Cash_Flow_df = Prod_df.copy(deep=True)
    
    Cash_Flow_df.loc[:, 'OPEX'] = OPEX  * Cash_Flow_df['BOEs'] * Cash_Flow_df['Days']
    
    Cash_Flow_df.loc[:, 'Ab'] = 0
    Cash_Flow_df.iloc[-1, Cash_Flow_df.columns.get_loc('Ab')] = decommissioning
    
    Cash_Flow_df.loc[:, 'Headco Cost'] = -Headco_cost
    
    Cash_Flow_df.loc[:, 'Gas Price'] = BTU_scf * USD_MMBTU
    Cash_Flow_df.loc[:, 'Oil Price'] = Oil_Price
    
    Cash_Flow_df.loc[:, 'Gas Rev'] = Cash_Flow_df['Gas Sale'] * Cash_Flow_df['Gas Price']
    Cash_Flow_df.loc[:, 'Liquids Rev'] = (Cash_Flow_df['Oil']+Cash_Flow_df['Gasoline']) * Cash_Flow_df['Oil Price']
    
    Cash_Flow_df.loc[:, 'Total Rev USD_day'] = Cash_Flow_df['Gas Rev']+Cash_Flow_df['Liquids Rev']
    Cash_Flow_df.loc[:, 'Total Revenue year'] = Cash_Flow_df['Total Rev USD_day'] * Cash_Flow_df['Days']
    Cash_Flow_df.loc[:, 'Tax'] = Cash_Flow_df['Total Revenue year'] * -Gov_tax
    
    Cash_Flow_df.loc[:, 'EBITDA'] = Cash_Flow_df['Total Revenue year'] + Cash_Flow_df['OPEX'] + Cash_Flow_df['CAPEX'] 
    
    Cash_Flow_df.loc[:, 'Operating Cash Flow'] = Cash_Flow_df['EBITDA'] + Cash_Flow_df['Tax'] + Cash_Flow_df['Ab'] 
    
    Cash_Flow_df.loc[:, 'Free Cash Flow'] = Cash_Flow_df['Operating Cash Flow'] + Cash_Flow_df['Ab'] 
    
    pd.set_option('display.float_format', '{:.2f}'.format)

    discount_rate = 0.10  # 10% cost of capital
    cash_flows_L = Cash_Flow_df['Free Cash Flow'].tolist()
    #print(cash_flows_L)

    npv_result = npf.npv(discount_rate, cash_flows_L)    # Compute the NPV # The function automatically treats cash_flows[0] as Year 0 (undiscounted)

    # -> Check if Cash Flow has negative and positive values before IRR is calculated
    has_pos = (Cash_Flow_df['Free Cash Flow'] > 0).any()
    has_neg = (Cash_Flow_df['Free Cash Flow'] < 0).any()

    if has_pos == True and has_neg == True:
        irr = pyxirr.irr(cash_flows_L)                       # Calculate IRR
        print(irr)
        if irr is None: irr = 0
    else:
        irr = 0
        
    # -> Display Results
    print(f"Gas Price: ${GasPrice:,.2f}")
    print(f"Oil Price: ${OilPrice:,.2f}") 

    print('---------------------------')
    print(f"Project NPV: ${npv_result:,.2f}") # Display the formatted result
    
    print(f"Internal Rate of Return: {irr:.3f}")
    
    Cash_Flow_df.head(4)
    display(Cash_Flow_df)
    
    specific_sum = Cash_Flow_df[['CAPEX', 'Free Cash Flow']].sum()
    display(specific_sum)

    return npv_result, irr


In [1]:
def Plot_Input_func(Input_var_dict):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    fig, AX = plt.subplots(nrows = 4, ncols=4, figsize=(15,10), layout ='constrained')
    
    #cmap = matplotlib.cm.get_cmap('rocket')
    cmap = sns.color_palette('Set2', as_cmap=True)
    
    icolor = 0.1
    i_samples = 10000
    i_col = 0
    i_row = 0
    for key in Input_var_dict.keys():
        
        RBG = cmap(icolor)[0],cmap(icolor)[1],cmap(icolor)[2]
        icolor = icolor + 0.1
        
        i_mean, i_SD = Input_var_dict[key][2], Input_var_dict[key][3]
        i_Low_limit, i_High_limit = Input_var_dict[key][0], Input_var_dict[key][1]
        i_constant = Input_var_dict[key][4]
        i_x_Label, i_Title = Input_var_dict[key][6], Input_var_dict[key][6]
        i_Dist_Type = Input_var_dict[key][5]
    
        Generate_normal_and_Truncated(i_mean, i_SD, i_samples, i_Low_limit, i_High_limit, i_constant, i_x_Label, i_Title, i_col,i_row,AX,RBG,i_Dist_Type)
        
        i_col = i_col + 1
        if i_col > 3:
            i_col = 0
            i_row = 2

In [5]:
def Generate_normal_and_Truncated(V_mu,V_SD,V_samples,V_Low_limit,V_High_limit,V_Constant, V_x_Label, V_Title, n_col,n_row,AX,RBG, Dist_Type=None):
    import numpy as np
    import scipy.stats as stats
    
    n_bins = 30
    N_Samples = np.random.normal(loc = V_mu, scale = V_SD, size= V_samples) # normal distribution

    if Dist_Type == 'Constant':
        N_Samples = np.random.uniform(low = V_Constant, high = V_Constant, size = 10000)
        
    if Dist_Type == 'Uniform':
        N_Samples = np.random.uniform(low = V_Low_limit, high = V_High_limit, size = 10000)
        
    if Dist_Type == 'Normal':
        N_Samples = np.random.normal(loc = V_mu, scale = V_SD, size= V_samples) # normal distribution

    if Dist_Type == 'Triangular':
        N_Samples = np.random.triangular(left=V_Low_limit, mode=V_mu,right= V_High_limit, size=10000)
        
    # Plot Normal Distribution
    AX[n_row,n_col].hist(N_Samples, bins=n_bins, density=True, align='mid', color=(RBG), alpha=1, edgecolor = "black")
    AX[n_row,n_col].set_xlabel(' ')
    AX[n_row,n_col].set_ylabel('Probability Density')
    AX[n_row,n_col].set_title(V_Title, weight='bold', fontsize = 18)
    x_axis = AX[n_row,n_col].set_xlim()

    
    ## generate data for truncated normal distribution
    
    NT_Samples = stats.truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)

    if Dist_Type == 'Constant':
        NT_Samples = np.random.uniform(low = V_Constant, high = V_Constant, size = 10000)
        
    if Dist_Type == 'Uniform':
        NT_Samples = np.random.uniform(low = V_Low_limit, high = V_High_limit, size = 10000)
        
    if Dist_Type == 'Normal':
        NT_Samples = stats.truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)
        
    ## Plot truncated Normal distribution
    AX[n_row+1,n_col].hist(NT_Samples, bins=n_bins, density=True, align='mid', color=RBG, alpha=0.4, edgecolor = "black")
    AX[n_row+1,n_col].set_xlabel(' ')
    AX[n_row+1,n_col].set_ylabel('PD')
    AX[n_row+1,n_col].set_title(V_Title + ' (TRUNCATED)')
    AX[n_row+1,n_col].set_xlim(x_axis)

### Function to calculate NPV IRR

In [ ]:
def Calc_Eco_Eval(Prod_df):

    BTU_scf = sheet["D29"].value
    USD_MMBTU = sheet["D30"].value
    Oil_Price = sheet["D32"].value
    Gov_tax = sheet["D35"].value
    Company_Share = sheet["D36"].value
    Headco_cost = sheet["D38"].value*1000000
    OPEX = sheet["D37"].value
    decommissioning = sheet["D26"].value

    Cash_Flow_df = pd.DataFrame()
    Cash_Flow_df = Prod_df.copy(deep=True)
    
    Cash_Flow_df.loc[:, 'OPEX'] = OPEX  * Cash_Flow_df['BOEs'] * Cash_Flow_df['Days']
    
    Cash_Flow_df.loc[:, 'Ab'] = 0
    Cash_Flow_df.iloc[-1, Cash_Flow_df.columns.get_loc('Ab')] = decommissioning
    
    Cash_Flow_df.loc[:, 'Headco Cost'] = -Headco_cost
    
    Cash_Flow_df.loc[:, 'Gas Price'] = BTU_scf * USD_MMBTU
    Cash_Flow_df.loc[:, 'Oil Price'] = Oil_Price
    
    Cash_Flow_df.loc[:, 'Gas Rev'] = Cash_Flow_df['Gas Sale'] * Cash_Flow_df['Gas Price']
    Cash_Flow_df.loc[:, 'Liquids Rev'] = (Cash_Flow_df['Oil']+Cash_Flow_df['Gasoline']) * Cash_Flow_df['Oil Price']
    
    Cash_Flow_df.loc[:, 'Total Rev USD_day'] = Cash_Flow_df['Gas Rev']+Cash_Flow_df['Liquids Rev']
    Cash_Flow_df.loc[:, 'Total Revenue year'] = Cash_Flow_df['Total Rev USD_day'] * Cash_Flow_df['Days']
    Cash_Flow_df.loc[:, 'Tax'] = Cash_Flow_df['Total Revenue year'] * -Gov_tax
    
    Cash_Flow_df.loc[:, 'EBITDA'] = Cash_Flow_df['Total Revenue year'] + Cash_Flow_df['OPEX'] + Cash_Flow_df['CAPEX'] 
    
    Cash_Flow_df.loc[:, 'Operating Cash Flow'] = Cash_Flow_df['EBITDA'] + Cash_Flow_df['Tax'] + Cash_Flow_df['Ab'] 
    
    Cash_Flow_df.loc[:, 'Free Cash Flow'] = Cash_Flow_df['Operating Cash Flow'] + Cash_Flow_df['Ab'] 
    
    pd.set_option('display.float_format', '{:.2f}'.format)
    
    
    
    discount_rate = 0.10  # 10% cost of capital
    cash_flows_L = Cash_Flow_df['Free Cash Flow'].tolist()
    #print(cash_flows_L)
    
    # Compute the NPV
    # The function automatically treats cash_flows[0] as Year 0 (undiscounted)
    npv_result = npf.npv(discount_rate, cash_flows_L)
    print('---------------------------')
    print(f"Project NPV: ${npv_result:,.2f}") # Display the formatted result
    
    # Calculate IRR
    irr = pyxirr.irr(cash_flows_L)
    print(f"Internal Rate of Return: {irr:.3f}")
    
    Cash_Flow_df.head(4)
    display(Cash_Flow_df)
    
    specific_sum = Cash_Flow_df[['CAPEX', 'Free Cash Flow']].sum()
    display(specific_sum)


### Plot Distributions

In [2]:
def Plot_Dist_Cum(OGIP_Samples,r_chart, x_Label, ax2):
    if r_chart == 0:
        alp = 0.5
    else:
        alp = 0.8
    N, bins, patches = ax2[r_chart,0].hist(OGIP_Samples, bins=60, edgecolor='black', linewidth=1, color='red',alpha=alp)
    
    # facecolor for each bar
    pn90 = round(np.percentile(OGIP_Samples, 10),3)
    pn10 = round(np.percentile(OGIP_Samples, 90),3)
    for i in range(len(N)):
        if (bins[i] < pn90) or (bins[i]>=pn10):
           patches[i].set_facecolor('yellow')

    #ax2[r_chart,0].set_xlim(left=0)
    ax2[r_chart,0].set_xlabel(x_Label, fontsize = 18,weight='bold');

    y_limit = ax2[r_chart,0].get_ylim()
    ax2[r_chart,0].set_ylim(y_limit)

    ### Plot black dot Mean ###
    n_Mean = round(np.mean(OGIP_Samples),3)
    if n_Mean > 1:
        n_Mean = round(n_Mean,0)
        
    n_txt = str(n_Mean)
    ax2[r_chart,0].annotate(n_txt, xy=(n_Mean,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                             horizontalalignment='center')

    ax2[r_chart,0].annotate('Mean', xy=(n_Mean,y_limit[1]), xycoords='data', xytext=(0,17), textcoords='offset points',
                             horizontalalignment='center')
        
    ax2[r_chart,0].scatter(n_Mean, y_limit[1], clip_on=False,color='black')


    ### Plot black dot P90 ###
    pn = round(np.percentile(OGIP_Samples, 10),3)
    if pn > 1:
        pn = round(pn,0)
        
    n_txt = str(pn)
    ax2[r_chart,0].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                            horizontalalignment='center')

    ax2[r_chart,0].annotate('P90', xy=(pn,y_limit[1]), xycoords='data', xytext=(0,17), textcoords='offset points',
                            horizontalalignment='center')
        
    ax2[r_chart,0].scatter(pn, y_limit[1], clip_on=False,color='black') 
    
    ### Plot black dot P10 ###
    pn = round(np.percentile(OGIP_Samples, 90),3)
    max_val = np.max(OGIP_Samples)

    if pn > 1:
        pn = round(pn,1)
        
    n_txt = str(pn)
    ax2[r_chart,0].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,5), textcoords='offset points',
                            horizontalalignment='center')
    n_txt = 'P10'
    ax2[r_chart,0].annotate(n_txt, xy=(pn,y_limit[1]), xycoords='data', xytext=(0,17), textcoords='offset points',
                            horizontalalignment='center')
        
    x_val = pn
    y_val = y_limit[1]
    ax2[r_chart,0].scatter(x_val, y_val, clip_on=False,color='black')
    
    
    
    #########################################################
    # Cumulative Plot
    Nc, binsc, patchesc = ax2[r_chart,1].hist(OGIP_Samples,cumulative=-1, bins=60, edgecolor='black', linewidth=1,color='red',alpha=alp)
    axx2 = ax2[r_chart,1]
    # facecolor for each bar
    pn90, pn10 = round(np.percentile(OGIP_Samples, 10),4), round(np.percentile(OGIP_Samples, 90),4)
    max_ogip = np.max(binsc)
    
    for i in range(len(Nc)):
        if (binsc[i] < pn90) or (binsc[i]>=pn10):
           patchesc[i].set_facecolor('yellow')
        
        if binsc[i]<n_Mean:
            y_limit_M=Nc[i]

        if binsc[i]<pn90:
            y_limit_P90=Nc[i]

        if binsc[i]<pn10:
            y_limit_P10=Nc[i]
    
    #axx2.set_xlim(left=0)
    axx2.set_xlabel(x_Label, fontsize = 18,weight='bold');
    
    dx = max_ogip/60
    x_vals, y_vals = [n_Mean, pn90+dx, pn10+dx], [y_limit_M,y_limit_P90,y_limit_P10]
    
    axx2.scatter(x_vals, y_vals, clip_on=False,color='black') # add dots for Mean, P90 and P10
    
    # P10 Annotations on plot
    Add_annotations('P10',pn10, y_limit_P10, dx,axx2)
    Add_annotations('Mean',n_Mean, y_limit_M, dx,axx2)
    Add_annotations('P90',pn90, y_limit_P90, dx,axx2)

In [3]:
def Add_annotations(n_txt,x_val,y_val,dx,axx2):
    xx_val,yy_val = x_val+(dx*0), y_val
    axx2.annotate(n_txt, xy=(xx_val,yy_val), xycoords='data', xytext=(0,5), textcoords='offset pixels',
                  horizontalalignment='left', fontsize=9)
    
    n_txt,xx_val, yy_val = str(round(x_val,2)), x_val+(dx*0), y_val
    axx2.annotate(n_txt, xy=(xx_val,yy_val), xycoords='data', xytext=(0,15), textcoords='offset pixels',
                  horizontalalignment='left', fontsize=14, weight='bold')